# Bonus: The Agent Loop with an Embedding Retriever

Sections 02 through 06 kept the retriever fixed as a pure-Python TF-IDF implementation so that every improvement could be attributed to the agent loop. This bonus section puts that claim under pressure.

We swap the retriever for ChromaDB backed by `ibm-granite/granite-embedding-30m-english` (the same embedding model the Escalation Lab used) and run the *same* agent loop over the *same* 10 evaluation questions. Two questions drive the exercise:

1. **Does the architectural pattern generalize?** The agent loop should work over any retriever that returns ranked text chunks. This notebook proves it by swapping only the retriever and changing nothing else.
2. **Was the TF-IDF choice doing the loop any favors?** If embeddings fix failures the loop could not, the core lab's isolation argument is weakened. If results are identical, the core lab's design holds up.

The answer turns out to matter for your production decisions, not just this lab.

## What this notebook needs that the rest of the lab does not

This is the only notebook in the lab that reaches outside the MaaS endpoint for dependencies:

- **`chromadb`** and **`sentence-transformers`** (installed in the cell below, scoped to this notebook).
- **First-run Hugging Face access** to download `ibm-granite/granite-embedding-30m-english` (~61 MB, cached after the first run).
- The persisted Chroma collection at `../prebuilt/chroma_agentic/` (~800 KB, committed with the repo).

**If you are on a locked-down environment with no outbound HF access**, this notebook degrades gracefully: it detects the failure and loads pre-recorded results from `../prebuilt/agent_loop_embeddings_results.json`, so you can still read through the comparison below. A banner further down will tell you which mode you ended up in.

In [ ]:
! pip install -q chromadb sentence-transformers pysqlite3-binary

## 7.1 Setup and Mode Detection

We try to load the Chroma retriever first. If the embedding model cannot be fetched (no network, no HF access, missing deps), we fall back to pre-recorded results so the rest of the notebook still runs. `MODE` is the one variable downstream cells branch on.

In [ ]:
import sys, os, json
from pathlib import Path

# Make utils/ importable from the project root
project_root = Path('..').resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import config
from openai import OpenAI

client = OpenAI(api_key=config.API_KEY, base_url=config.ENDPOINT_BASE)

MODE = None
collection = None
load_error = None

try:
    from utils.retriever_chroma import ChromaRetriever
    collection = ChromaRetriever.load()
    assert collection.count() == 30, f'Expected 30 chunks, got {collection.count()}'
    MODE = 'live'
except Exception as e:
    MODE = 'fallback'
    load_error = repr(e)

if MODE == 'live':
    print('Mode: LIVE')
    print(f'  Retriever   : {collection.name}')
    print(f'  Chunks      : {collection.count()}')
    print(f'  Model       : {config.MODEL_ID}')
else:
    print('Mode: FALLBACK (pre-recorded results)')
    print(f'  Reason      : {load_error}')
    print('  Downstream cells will load prebuilt/agent_loop_embeddings_results.json.')

## 7.2 The Only Thing That Changed

In Section 2.1.2 we loaded a TF-IDF retriever:

```python
from utils.retriever import Retriever
collection = Retriever.load()
```

In Section 7.1 above we loaded a Chroma retriever:

```python
from utils.retriever_chroma import ChromaRetriever
collection = ChromaRetriever.load()
```

Both expose the same `.query()` shape:

```python
collection.query(query_texts=[q], n_results=3, include=['documents', 'distances'])
# returns {'documents': [[...]], 'distances': [[...]]}
```

So the tool implementation from Section 3, the agent loop from Section 4, and the evaluator from Section 5 all take this new `collection` without any code change. The cell below demonstrates a single embedding-backed query to confirm the interface is honored.

In [ ]:
if MODE == 'live':
    res = collection.query(
        query_texts=['How does a Thief attempt to pick a lock?'],
        n_results=3,
        include=['documents', 'distances'],
    )
    for doc, dist in zip(res['documents'][0], res['distances'][0]):
        print(f'[{dist:.4f}] {doc[:140]}...')
else:
    print('(skipped in FALLBACK mode; see prebuilt results below)')

## 7.3 The Loop Is The Same

The two cells below are **copied verbatim** from Section 4 (`04_Running_the_Agent_Loop.ipynb`, cells 4.4 and 4.5). No edits. That is the pedagogical point: the control structure we defined in the core lab works over any retriever that honors the `.query()` contract. The retriever is an injected dependency, not a design assumption.

If you want to convince yourself nothing was changed, diff this against Section 4.

In [ ]:
import ast, operator

# --- Tool implementations (verbatim from Section 3, using shared retriever) ---

def rag_retrieval(query: str) -> dict:
    """Search the Basic Fantasy RPG corpus for relevant chunks."""
    results = collection.query(
        query_texts=[query],
        n_results=3,
        include=['documents', 'distances'],
    )
    chunks = []
    for doc, dist in zip(results['documents'][0], results['distances'][0]):
        chunks.append({'text': doc, 'distance': round(dist, 4)})
    return {'tool': 'rag_retrieval', 'query': query, 'chunks': chunks}


_SAFE_OPS = {
    ast.Add: operator.add, ast.Sub: operator.sub,
    ast.Mult: operator.mul, ast.Div: operator.truediv,
    ast.FloorDiv: operator.floordiv, ast.Mod: operator.mod,
    ast.Pow: operator.pow, ast.USub: operator.neg,
}

def _safe_eval_node(node):
    if isinstance(node, ast.Expression):
        return _safe_eval_node(node.body)
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _SAFE_OPS:
        return _SAFE_OPS[type(node.op)](_safe_eval_node(node.left), _safe_eval_node(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _SAFE_OPS:
        return _SAFE_OPS[type(node.op)](_safe_eval_node(node.operand))
    raise ValueError(f'Unsupported operation: {ast.dump(node)}')

def calculator(expression: str) -> dict:
    try:
        tree = ast.parse(expression, mode='eval')
        return {'tool': 'calculator', 'expression': expression, 'result': _safe_eval_node(tree)}
    except (ValueError, SyntaxError, TypeError, ZeroDivisionError) as e:
        return {'tool': 'calculator', 'expression': expression, 'error': str(e)}

def no_answer() -> dict:
    return {
        'answer': 'I do not have enough information to answer this question from the available corpus.',
        'tool': 'no_answer',
    }

TOOL_DISPATCH = {'rag_retrieval': rag_retrieval, 'calculator': calculator, 'no_answer': no_answer}

def execute_tool(name, arguments):
    if name not in TOOL_DISPATCH:
        return json.dumps({'error': f'Unknown tool: {name}'})
    return json.dumps(TOOL_DISPATCH[name](**arguments), default=str)

with open('../prebuilt/tool_definitions.json') as f:
    tool_definitions = json.load(f)['tool_definitions']

print(f'Tools registered: {list(TOOL_DISPATCH.keys())}')

In [ ]:
# Verbatim from Section 4.5 (04_Running_the_Agent_Loop.ipynb, cell 10).
def run_agent_loop(question, tools, client, model_id, max_iterations=3, verbose=False):
    system_prompt = (
        'You are a rules assistant for Basic Fantasy RPG. '
        'Use the available tools to answer questions about the game rules. '
        'If the retrieved context is sufficient, answer the question directly. '
        'If the context is insufficient or irrelevant, use the no_answer tool. '
        'If the question requires a calculation, use the calculator tool. '
        'Always base your answer on tool results, not prior knowledge.'
    )
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': question},
    ]
    trace = {'question': question, 'steps': [], 'final_answer': None, 'iterations': 0}

    for iteration in range(1, max_iterations + 1):
        trace['iterations'] = iteration
        response = client.chat.completions.create(
            model=model_id, messages=messages, tools=tools, temperature=0.0,
        )
        choice = response.choices[0]
        if choice.message.tool_calls:
            messages.append(choice.message)
            for tc in choice.message.tool_calls:
                name = tc.function.name
                args = json.loads(tc.function.arguments) if tc.function.arguments else {}
                result = execute_tool(name, args)
                trace['steps'].append({
                    'iteration': iteration, 'tool': name, 'arguments': args,
                    'result': json.loads(result),
                })
                messages.append({'role': 'tool', 'tool_call_id': tc.id, 'content': result})
        else:
            trace['final_answer'] = choice.message.content
            return trace

    messages.append({
        'role': 'user',
        'content': 'Based on the tool results above, provide your final answer to the original question.',
    })
    response = client.chat.completions.create(model=model_id, messages=messages, temperature=0.0)
    trace['final_answer'] = response.choices[0].message.content
    return trace

print('Agent loop ready.')

## 7.4 Run All 10 Questions

In LIVE mode this runs the loop against the MaaS endpoint, then scores each answer with the same judge prompt used in Section 4. In FALLBACK mode it loads the pre-recorded run. In both cases, the output ends up in `agent_loop_results` as a list of 10 entries, one per eval question.

In [ ]:
JUDGE_PROMPT = """You are an evaluation judge. Compare the EXPECTED answer to the ACTUAL answer.

The ACTUAL answer is correct if it conveys the same key facts as the EXPECTED answer,
even if the wording differs. Minor omissions of non-essential details are acceptable.

Respond with EXACTLY one JSON object:
{\"classification\": \"pass\" or \"fail\", \"reason\": \"<one sentence>\"}
"""

def judge(question, expected, actual):
    response = client.chat.completions.create(
        model=config.MODEL_ID,
        messages=[
            {'role': 'system', 'content': JUDGE_PROMPT},
            {'role': 'user', 'content': f'QUESTION: {question}\n\nEXPECTED: {expected}\n\nACTUAL: {actual}'},
        ],
        temperature=0.0,
    )
    raw = response.choices[0].message.content.strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {'classification': 'error', 'reason': 'Could not parse judge response'}

if MODE == 'live':
    with open('../prebuilt/eval_results.json') as f:
        eval_data = json.load(f)

    agent_loop_results = []
    for r in eval_data['results']:
        print(f"[{r['id']}] {r['question'][:70]}")
        trace = run_agent_loop(r['question'], tool_definitions, client, config.MODEL_ID, max_iterations=3)
        judgment = judge(r['question'], r['expected'], trace['final_answer'])
        agent_loop_results.append({
            'id': r['id'], 'question': r['question'], 'expected': r['expected'],
            'category': r['category'], 'passive_classification': r['classification'],
            'agent_answer': trace['final_answer'], 'iterations': trace['iterations'],
            'tools_used': [s['tool'] for s in trace['steps']],
            'trace': trace['steps'],
            'agent_classification': judgment['classification'],
            'judge_reason': judgment.get('reason', ''),
        })
        print(f"   -> {judgment['classification']}  iters={trace['iterations']}")
else:
    with open('../prebuilt/agent_loop_embeddings_results.json') as f:
        agent_loop_results = json.load(f)['results']
    for r in agent_loop_results:
        print(f"[{r['id']}] {r['agent_classification']}  iters={r['iterations']}  tools={r['tools_used']}")

passes = sum(1 for ar in agent_loop_results if ar['agent_classification'] == 'pass')
print(f"\nEmbedding-backed agent loop: {passes}/10 passes")

## 7.5 Side-by-Side: TF-IDF vs Embeddings

Now the real question. Section 4 saved TF-IDF agent-loop results to `prebuilt/agent_loop_results.json`. We load them and put the two runs next to each other, looking specifically for:

- **Total pass count.** If embeddings unlock failures TF-IDF missed, the score goes up.
- **Which questions each retriever passes.** Same pass set, or different?
- **Tool sequences.** Did either retriever trigger a query rewrite (more than one `rag_retrieval` call)? That would be evidence of the evaluator stepping in to compensate for weak retrieval.
- **Same-category failures.** A failure that persists across retrievers is probably architectural, not retrieval-quality.


In [ ]:
with open('../prebuilt/agent_loop_results.json') as f:
    tfidf_results = json.load(f)['results']

emb_by_id = {ar['id']: ar for ar in agent_loop_results}

print(f"{'ID':<5} {'TF-IDF':<24} {'Embedding':<24} {'Question':<40}")
print('-' * 100)
for t in tfidf_results:
    e = emb_by_id[t['id']]
    t_tools = '+'.join(t['tools_used']) or 'none'
    e_tools = '+'.join(e['tools_used']) or 'none'
    t_cell = f"{t['agent_classification']:<5} {t['iterations']}i {t_tools}"
    e_cell = f"{e['agent_classification']:<5} {e['iterations']}i {e_tools}"
    print(f"{t['id']:<5} {t_cell:<24} {e_cell:<24} {t['question'][:38]}")

t_pass = sum(1 for r in tfidf_results if r['agent_classification'] == 'pass')
e_pass = sum(1 for r in agent_loop_results if r['agent_classification'] == 'pass')

# Count rewrites: any question with more than one rag_retrieval call
t_rewrites = sum(1 for r in tfidf_results if sum(1 for s in r['trace'] if s['tool'] == 'rag_retrieval') > 1)
e_rewrites = sum(1 for r in agent_loop_results if sum(1 for s in r['trace'] if s['tool'] == 'rag_retrieval') > 1)

print('-' * 100)
print(f"Passes      : TF-IDF {t_pass}/10   Embedding {e_pass}/10")
print(f"Rewrites    : TF-IDF {t_rewrites}/10  Embedding {e_rewrites}/10  (questions that required a second retrieval)")

t_fails = {r['id'] for r in tfidf_results if r['agent_classification'] != 'pass'}
e_fails = {r['id'] for r in agent_loop_results if r['agent_classification'] != 'pass'}
print(f"TF-IDF fails on    : {sorted(t_fails) or 'none'}")
print(f"Embedding fails on : {sorted(e_fails) or 'none'}")
print(f"Shared failures    : {sorted(t_fails & e_fails) or 'none'}")

## 7.6 What This Tells Us

The empirical result on this corpus is striking: **both retrievers score identically, fail on the same question, and neither triggered a query rewrite.** The failing question is a calculator failure, not a retrieval failure; no retriever upgrade could have rescued it because the chunk was retrieved correctly in both runs; the model just produced the wrong arithmetic output from the correct context.

Three conclusions follow, in order of importance:

### 1. The architectural pattern generalizes (the lab's central claim holds)

We changed the retriever and nothing else. The loop, the tools, the prompts, the evaluation, all unchanged. The loop ran, scored the same, and behaved the same. That is the strongest possible evidence that the agent loop is a genuine architectural pattern, not a technique that was accidentally co-designed with a particular retriever.

### 2. The 'retriever was doing the loop no favors' concern was unfounded on this corpus

One legitimate worry about the core lab's design was that the query-rewrite step might have been silently compensating for a weak retriever, which would mean part of the loop's apparent win was really the loop paying interest on the TF-IDF tax. The comparison above rules that out for this lab: **zero rewrites in either run**. The loop's value here came from the `evaluate` and `abstain` capabilities, not from `rewrite`. The retriever choice was, empirically, neutral.

### 3. That neutrality is corpus-specific; do not generalize it to production

The Basic Fantasy RPG rulebook is the ideal case for TF-IDF: small, lexically consistent, with stable proper nouns that users actually type (class names, spell names, rule titles). Most production corpora are not like this:

- Users paraphrase. *"How do I heal my party?"* will not lexically match a chunk titled *Cure Light Wounds*.
- Domains drift. Technical documentation, medical notes, and customer support tickets all have heavy synonymy and vocabulary mismatch between queries and source text.
- Multilingual or multimodal corpora are out of reach for TF-IDF entirely.

In any of those settings, the rewrite step (or a better retriever) starts doing real work, and the comparison above would look different. What this bonus section tells you is that the **pattern** you learned in the core lab is the right one; it is the **production knob for retriever choice** that stays open.

### 4. Where to spend your next hour in a real deployment

If you are adapting this lab to your own corpus and your pass rate is stuck:

- **Look at the rewrites first.** A high rewrite rate is your retriever telling you its default is too weak for the query distribution. Swap to embeddings.
- **Look at the failures after that.** If every retriever finds the right chunk and the loop still fails, the problem is upstream (chunking) or downstream (generation grounding, calculator-style reasoning). No retriever change will fix it. This is what `q06` looks like in the table above.
- **Treat the retriever as a pluggable component from day one.** The fact that swapping `Retriever` for `ChromaRetriever` was a one-line import is not an accident of this lab; it is the interface discipline that keeps the loop honest.

---

> **Back to the core lab:** Section 2.1.2 explained *why* we chose TF-IDF for the main path. This bonus section is the experimental check on that argument. If your own adaptation of the lab runs into vocabulary mismatch between queries and corpus, the `ChromaRetriever` class in `utils/retriever_chroma.py` is a drop-in replacement; only the import changes.